In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
target_df=spark.table('voltgrid.silver.customers')

In [0]:
df=spark.table('voltgrid.bronze.customers')

In [0]:
df=df.withColumn("rnk",F.row_number().over(Window.partitionBy('customer_id').orderBy(F.desc('ingestion_time'))))\
    .filter(F.col('rnk')==1).drop('rnk','ingestion_time','source_path')

In [0]:
history_records = target_df.filter(F.col('active_flg') == 'N')

In [0]:
source_df=df.withColumn('email',F.lower('email'))\
    .withColumn('full_name',F.upper('full_name'))\
        .withColumn('loyalty_tier',F.upper('loyalty_tier'))

## **`SCD Type2` Start From Here**

In [0]:
target_active_records=target_df.filter(F.col("active_flg")=="Y")

In [0]:
hash_keys=['customer_id','full_name','email','phone','loyalty_tier']

In [0]:
hash_keys=['customer_id','full_name','email','phone','loyalty_tier']
source_df=source_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))
target_active_records=target_df.filter(F.col("active_flg")=="Y").withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))

### `changed_records` having only records which are changed in source table     

In [0]:
changed_records=target_active_records.alias('t').join(source_df.alias('s'),F.col("t.customer_id")==F.col("s.customer_id"),'inner').filter(F.col("t.hash_key") != F.col("s.hash_key")).select(F.col("t.*")).withColumn('active_flg',F.lit('N')).withColumn('end_date',F.current_date()).drop('hash_key')

In [0]:
target_active_records=target_active_records.drop("hash_key")
source_df=source_df.drop("hash_key")

##  `new_records` Only new records will exist

In [0]:
new_records=(source_df.alias("s")\
    .join(target_active_records.alias("t"),F.col("s.customer_id")==F.col("t.customer_id"),'left_anti')\
        .withColumn('active_flg',F.lit('Y'))\
            .withColumn('start_date',F.current_date())\
                .withColumn('end_date',F.lit(None).cast('date')))

## `old_records` have records which are not changed

In [0]:
old_records=target_active_records.alias("t").join(changed_records.alias("s"),F.col("s.customer_id")==F.col("t.customer_id"),'left_anti')

## `changed_records_value` Records which are changed

In [0]:
changed_records_value=source_df.alias('src').join(changed_records.alias('chg'),F.col('src.customer_id')==F.col('chg.customer_id'),'left_semi').withColumn('active_flg', F.lit('Y')).withColumn('start_date', F.current_date()).withColumn('end_date', F.lit(None).cast('date'))

In [0]:
total_records = (
    history_records.unionByName(old_records)
    .unionByName(new_records)
    .unionByName(changed_records)
    .unionByName(changed_records_value)
)

In [0]:
total_records.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("voltgrid.silver.customers")